# 05b · Paper figures (TRUST 2027): polished, from the saved CSVs

**What this step does.** Redraws every exhibit for the paper from the result CSVs. Nothing is recomputed,
so the figures always match the numbers in `results/`. Each figure is written twice to `figures/paper/`:
a vector **PDF** (put this one in the LaTeX source) and a 300 dpi **PNG** (for slides, drafts, Word).

**Two renderers.**
* **Graphviz** draws the diagrams: the study pipeline and the Stage 0 causal graphs.
* **matplotlib** draws the data plots (lines and bars).

**Sized for ACM `sigconf`.** One column = 3.33 in, full page width (`figure*`) = 7.0 in. Fonts are 7-8 pt at
printed size and are embedded as TrueType (ACM's PDF checker rejects Type 3 fonts).

| File stem | Shows | Where in the paper | Width |
|---|---|---|---|
| `pipeline` | The five steps of the study (Graphviz) | Methods | full |
| `stage0_dag_diabetes130`, `stage0_dag_cdcdiabetes`, `stage0_dag_taiwancredit` | What Stage 0 found: the label's causes and the risk group's paths into them (Graphviz) | Methods | full, as three subfigures |
| `dose_recovery` | Confident learning's overall error estimate vs. the injected amount | Results | full |
| `targeted_harm` | Group recall and group calibration fall; overall AUC does not | Results | full |
| `rate_check` | Rate check aimed at the risk group detects the damage; the same check on everyone does not | Results | full |
| `repair_recovery` | Five repair strategies: lost detection restored, AND the side effects on AUC and group calibration | Results | full |
| `appx_localize` | Which edge into the label moves under each noise type | Appendix | full |
| `appx_southgerman` | South German Credit (retired; kept as a record) | Appendix / supplement | full |



**Datasets in the main figures (Oct 10, 2026):** Diabetes 130, CDC Diabetes and Taiwan credit default. South German Credit is in the appendix only.

In [1]:
# --- 1. setup: paths, data, shared style -----------------------------------------------------------
import os, sys, textwrap
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

ROOT = Path.cwd()                                   # the label_integrity folder (where this notebook lives)
RESULTS = ROOT / "results"
STAGE0 = ROOT / "stage0_out"
OUT = ROOT / "figures" / "paper"
OUT.mkdir(parents=True, exist_ok=True)

det = pd.read_csv(RESULTS / "03_detect_summary.csv")          # one row per dataset x mechanism x dose x seed
ver = pd.read_csv(RESULTS / "04_verify.csv")                  # model scores on the clean test set
rec = pd.read_csv(RESULTS / "04_recovery_recall_stratum.csv") # recovery ratio per repair
loc = pd.read_csv(RESULTS / "03b_localize.csv")               # Stage 0 coefficients into the label, rerun on noisy labels

MAIN = ["diabetes130", "cdcdiabetes", "taiwancredit"]   # datasets in the main figures
NAMES = {"diabetes130": "Diabetes 130", "cdcdiabetes": "CDC Diabetes", "southgerman": "South German Credit",
         "taiwancredit": "Taiwan Credit"}
GROUP = {"diabetes130": "age 80+", "cdcdiabetes": "no coverage / couldn't afford a doctor",
         "southgerman": "age 25 or under",
         "taiwancredit": "credit limit ≤ NT$50,000"}
MECHS = ["uniform", "class_conditional", "targeted"]
MECH_NAME = {"uniform": "random (uniform)", "class_conditional": "class-conditional",
             "targeted": "targeted missed events"}
# colour-blind-safe (Paul Tol); targeted noise is the 'hero' series, so it gets the strongest colour.
# Each series also has its own marker and line style, so the figures still read in greyscale.
COL = {"uniform": "#4477AA", "class_conditional": "#999933", "targeted": "#CC3311"}
MRK = {"uniform": "o", "class_conditional": "s", "targeted": "^"}
LS = {"uniform": (0, (4, 1.5)), "class_conditional": (0, (1, 1.2)), "targeted": "-"}
GREY = "#777777"
COL_W, PAGE_W = 3.33, 7.0                           # ACM sigconf column / text width, inches

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 7.5, "axes.titlesize": 8, "axes.labelsize": 7.5,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "xtick.major.size": 2.5, "ytick.major.size": 2.5,
    "axes.grid": True, "grid.color": "#E6E6E6", "grid.linewidth": 0.5, "axes.axisbelow": True,
    "lines.linewidth": 1.2, "lines.markersize": 3.2,
    "savefig.dpi": 300, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
    "pdf.fonttype": 42, "ps.fonttype": 42,          # embed TrueType, never Type 3
})


def savefig(fig, stem):
    for ext in ("pdf", "png"):
        fig.savefig(OUT / f"{stem}.{ext}", facecolor="white")
    plt.close(fig)
    print("saved", stem, "(.pdf, .png)")


def ci(s):
    """95% confidence half-width over seeds (normal approximation)."""
    return 1.96 * s.std(ddof=1) / np.sqrt(s.count()) if s.count() > 1 else 0.0


DOSE_LABEL = "label-error dose (share of eligible labels flipped)"
DCOL = {"diabetes130": "#88CCEE", "cdcdiabetes": "#332288", "taiwancredit": "#117733"}   # one color per dataset (bar charts)
print("rows:", {k: len(v) for k, v in dict(det=det, ver=ver, rec=rec, loc=loc).items()})

rows: {'det': 420, 'ver': 4920, 'rec': 4480, 'loc': 1260}


## 2. Graphviz setup
Graphviz is a separate program (it provides `dot`). On Windows its usual folder is added to PATH for this
notebook only. If the check below fails, install it once in PowerShell with `winget install graphviz`.

In [14]:
for d in (r"C:\Program Files\Graphviz\bin", r"C:\Program Files (x86)\Graphviz\bin"):
    if os.path.isdir(d) and d not in os.environ["PATH"]:
        os.environ["PATH"] += os.pathsep + d
import graphviz
print("Graphviz", ".".join(map(str, graphviz.version())))
GV_FONT = "Helvetica"                               # Graphviz maps this to Arial on Windows


def render_gv(g, stem):
    """Vector PDF straight from Graphviz, plus a 300 dpi PNG on a white background."""
    (OUT / f"{stem}.pdf").write_bytes(g.pipe(format="pdf"))     # pipe = no temporary .gv file left behind
    g.attr(dpi="300")
    (OUT / f"{stem}.png").write_bytes(g.pipe(format="png"))
    print("saved", stem, "(.pdf, .png)")

Graphviz 14.1.5


In [ ]:
#savefig(fig, "fig5_localize")

## Fig. pipeline · the study design (Graphviz)
Five steps left to right. Gold text marks every place the Stage 0 risk group is used, so the reader sees
that Stage 0 aims the noise, the audits, the review and the scoring.

In [15]:
RG = '<FONT COLOR="#8A6D00"><B>risk group</B></FONT>'      # gold = uses the group fixed by Stage 0


def step(title, body, fill, border):
    body = '<BR ALIGN="LEFT"/>'.join(b.replace("risk group", RG) for b in body) + '<BR ALIGN="LEFT"/>'
    return (f'<<TABLE BORDER="0" CELLPADDING="1" CELLSPACING="0">'
            f'<TR><TD ALIGN="LEFT"><B>{title}</B></TD></TR>'
            f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="8.5">{body}</FONT></TD></TR></TABLE>>'), fill, border


g = graphviz.Digraph("pipeline", engine="dot")
g.attr(rankdir="LR", nodesep="0.25", ranksep="0.28", pad="0.05", margin="0", bgcolor="white",
       fontname=GV_FONT, size="7,2.2",
       label='<<FONT POINT-SIZE="8.5"><FONT COLOR="#8A6D00"><B>Gold text</B></FONT>: '
             'uses the risk group that Stage 0 fixed before any noise was injected.</FONT>>',
       labelloc="b", labeljust="l")
g.attr("node", shape="box", style="rounded,filled", fontname=GV_FONT, fontsize="10",
       margin="0.10,0.06", penwidth="0.9", width="1.35", height="1.32")
g.attr("edge", fontname=GV_FONT, fontsize="8", arrowsize="0.6", penwidth="1.0", color="#444444")
steps = [
    ("s0", "Step 0 · Diagnose", ["Causal discovery on the", "training labels:", "the label's causes and",
                                  "paths from groups", "→ picks the risk group"], "#FFF1C2", "#8A6D00"),
    ("s1", "1 · Inject", ["Controlled label errors:", "random · class-conditional", "· targeted missed events",
                          "in the risk group", "7 doses × 5 seeds"], "#F2F2F2", "#888888"),
    ("s2", "2 · Detect", ["Confident learning on", "all records vs. risk group", "Rate check (400 re-verified",
                          "records): all vs. risk group", "Localize: rerun Stage 0", "on the noisy labels"],
     "#F2F2F2", "#888888"),
    ("s3", "3 · Repair", ["Automatic: prune · relabel", "· reweight", "Expert review of 5% of", "the risk group:",
                          "random vs. model-ranked", "(reviewer accuracy 0.8–1)"], "#F2F2F2", "#888888"),
    ("s4", "4 · Verify", ["Retrain; score on the", "clean test set:", "overall AUC,", "risk group recall,",
                          "risk group calibration"], "#E8EEF7", "#4477AA"),
]
for k, t, b, f, c in steps:
    lab, fill, border = step(t, b, f, c)
    g.node(k, lab, fillcolor=fill, color=border)
for a, b in [("s0", "s1"), ("s1", "s2"), ("s2", "s3"), ("s3", "s4")]:
    g.edge(a, b)
render_gv(g, "pipeline")

saved pipeline (.pdf, .png)


## Fig. Stage 0 causal graphs (Graphviz), one per main dataset
**What is drawn.** The label (black), every direct cause of the label that Step 0 found (grey), the protected
attributes (gold) and the variables that define the risk group (red outline). Edges into the label are always
shown with their standardized coefficient. Other edges are shown only when they start at a protected or
risk-group variable and |coefficient| ≥ `MIN_COEF`; edges among the label's causes are left out to keep the
figure readable (they are in the `_full` views from notebook 01b).

**The highlighted edge** (red) is the one that changes most when the targeted missed-event errors are
injected (appendix figure `appx_localize`): age → readmission weakens to zero in Diabetes 130; a new negative
edge from couldn't-afford-a-doctor appears in CDC Diabetes; in Taiwan the credit-limit edge into default is the
first to disappear (gone by the 5% dose).

In [16]:
import json
import networkx as nx

MIN_COEF = 0.05
PRETTY = {
    "race_black": "race: Black", "female": "female", "age_years": "age", "readmit30": "readmitted ≤30 days",
    "number_inpatient": "prior inpatient visits", "number_emergency": "prior ER visits",
    "number_outpatient": "prior outpatient visits", "num_medications": "# medications",
    "num_lab_procedures": "# lab procedures", "num_procedures": "# procedures", "time_in_hospital": "length of stay",
    "number_diagnoses": "# diagnoses", "a1c_tested": "HbA1c tested", "a1c_high": "HbA1c elevated",
    "insulin_any": "on insulin", "med_change": "medication changed", "diabetes_med": "diabetes medication",
    "emergency_admit": "emergency admission",
    "diabetes_any": "diabetes or prediabetes", "HighBP": "high blood pressure", "HighChol": "high cholesterol",
    "CholCheck": "cholesterol check (5 yr)", "BMI": "BMI", "HeartDiseaseorAttack": "heart disease / attack",
    "HvyAlcoholConsump": "heavy drinking", "GenHlth": "general health (poor)", "DiffWalk": "difficulty walking",
    "Age": "age", "Sex": "sex (male)", "Income": "income", "AnyHealthcare": "has health coverage",
    "NoDocbcCost": "couldn't afford a doctor", "MentHlth": "poor mental-health days",
    "PhysHlth": "poor physical-health days", "Education": "education", "Smoker": "smoker", "Stroke": "stroke",
    "PhysActivity": "physical activity", "Fruits": "eats fruit", "Veggies": "eats vegetables",
    "default_next": "default next month", "limit_bal": "credit limit", "married": "married",
    "education": "education level", "bill_amt1": "latest bill", "pay_amt1": "last payment",
    "pay_amt2": "payment 2 months ago", "pay_amt3": "payment 3 months ago",
    "pay_0": "late: last month", "pay_2": "late: 2 months ago", "pay_3": "late: 3 months ago",
    "pay_4": "late: 4 months ago", "pay_5": "late: 5 months ago", "pay_6": "late: 6 months ago",
}
DAGS = {  # tag, risk-group variables, the edge that moves under targeted noise (from, to, kind)
    "diabetes130": dict(tag="Diab", group=["age_years"], moved=("age_years", "readmit30", "weakens")),
    "cdcdiabetes": dict(tag="CDC", group=["AnyHealthcare", "NoDocbcCost"], moved=("NoDocbcCost", "diabetes_any", "appears")),
    "taiwancredit": dict(tag="TWN", group=["limit_bal"], moved=("limit_bal", "default_next", "weakens")),
}


def pretty(v, wrap=16):
    return textwrap.fill(PRETTY.get(v, v.replace("_", " ")), wrap)


def stage0_dag(name, spec):
    run = STAGE0 / name
    fp = json.loads((run / f"fingerprint_{spec['tag']}.json").read_text())
    label, protected, group = fp["label"], fp["protected"], spec["group"]
    e = pd.read_csv(run / f"edges_{spec['tag']}.csv")
    e = e[e.prob >= 0.5]
    parents = e[e.to == label].sort_values("coef_median", key=abs, ascending=False)
    sources = list(dict.fromkeys(protected + group))
    shown = set(sources) | {label} | set(parents["from"])
    side = e[e["from"].isin(sources) & e.to.isin(shown) & (e.to != label) & (e.coef_median.abs() >= MIN_COEF)]
    mu, mv, kind = spec["moved"]

    g = graphviz.Digraph(f"stage0_{name}", engine="dot")
    g.attr(rankdir="LR", splines="spline", nodesep="0.09", ranksep="0.55", pad="0.04", margin="0",
           bgcolor="white", fontname=GV_FONT, newrank="true")
    g.attr("node", shape="box", style="rounded,filled", fontname=GV_FONT, fontsize="10",
           margin="0.07,0.025", height="0.26", penwidth="0.8")
    g.attr("edge", arrowsize="0.5", fontname=GV_FONT, fontsize="8.5")
    for v in shown:
        if v == label:
            g.node(v, pretty(v, 14), fillcolor="#222222", color="black", fontcolor="white")
        elif v in group:
            g.node(v, pretty(v), fillcolor="#FCE4DE", color="#CC3311", penwidth="1.6",
                   fontcolor="black" if v not in protected else "black")
        elif v in protected:
            g.node(v, pretty(v), fillcolor="#FDDC7A", color="#8A6D00")
        else:
            g.node(v, pretty(v), fillcolor="#F2F2F2", color="#9A9A9A")
    with g.subgraph() as s:
        s.attr(rank="source")
        for v in sources:                           # protected and risk-group variables share the first rank
            s.node(v)
    with g.subgraph() as s:
        s.attr(rank="sink"); s.node(label)
    for _, r in side.iterrows():
        g.edge(r["from"], r["to"], color="#B9B9B9", penwidth=f"{0.5 + 4 * abs(r.coef_median):.2f}")
    for _, r in parents.iterrows():
        moved = (r["from"], label) == (mu, mv)
        g.edge(r["from"], label, label=f"{r.coef_median:+.2f}",
               color="#CC3311" if moved else "#333333", fontcolor="#CC3311" if moved else "#333333",
               penwidth="2.2" if moved else f"{0.6 + 6 * abs(r.coef_median):.2f}")
    if kind == "appears" and mu not in set(parents["from"]):
        g.edge(mu, mv, style="dashed", color="#CC3311", penwidth="1.6", constraint="false")
    render_gv(g, f"stage0_dag_{name}")
    return dict(nodes=len(shown), parent_edges=len(parents), side_edges=len(side))


for name, spec in DAGS.items():
    print(name, stage0_dag(name, spec))

saved stage0_dag_diabetes130 (.pdf, .png)
diabetes130 {'nodes': 7, 'parent_edges': 4, 'side_edges': 3}
saved stage0_dag_cdcdiabetes (.pdf, .png)
cdcdiabetes {'nodes': 14, 'parent_edges': 11, 'side_edges': 16}
saved stage0_dag_taiwancredit (.pdf, .png)
taiwancredit {'nodes': 12, 'parent_edges': 11, 'side_edges': 8}


## Fig. dose recovery · confident learning's overall estimate vs. the injected amount
**y:** confident learning's estimate of the label-error rate, minus its estimate on the clean data (its floor).
**x:** the share of *all* training labels actually flipped. On the dashed line the estimate would recover the
injected rate exactly. Mean ± 95% CI over 5 seeds.

In the health datasets the estimate tracks random errors but barely registers targeted missed events. In Taiwan it also tracks the targeted errors in aggregate (payment history makes a relabelled defaulter stand out), but it still cannot find which records they are (see the detection table in notebook 03).

In [17]:
base = det[det.dose == 0][["dataset", "mechanism", "seed", "est_noise"]]
h1 = det.merge(base, on=["dataset", "mechanism", "seed"], suffixes=("", "_0"))
h1["above"] = h1.est_noise - h1.est_noise_0


def dose_recovery_panel(ax, d, legend=False):
    g = h1[h1.dataset == d]
    lim = g.realized_global_fraction.max() * 1.05
    ax.plot([0, lim], [0, lim], ls="--", lw=0.7, color=GREY, zorder=1)
    for m in MECHS:
        a = (g[g.mechanism == m].groupby("dose")
             .agg(x=("realized_global_fraction", "mean"), y=("above", "mean"), e=("above", ci)).reset_index())
        ax.errorbar(a.x, a.y, yerr=a.e, color=COL[m], marker=MRK[m], ls=LS[m], capsize=1.5, elinewidth=0.6,
                    label=MECH_NAME[m], zorder=3)
    ax.set_title(NAMES[d])
    ax.set_xlabel("labels flipped (all)")
    ax.set_xlim(-0.005, lim); ax.set_ylim(-0.02, lim)
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))


fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 1.95), constrained_layout=True)
for ax, d in zip(axes, MAIN):
    dose_recovery_panel(ax, d)
axes[0].set_ylabel("estimated error rate\nabove clean floor")
h, l = axes[0].get_legend_handles_labels()
h.append(Line2D([], [], ls="--", lw=0.7, color=GREY)); l.append("perfect recovery")
fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4, frameon=False, handlelength=2.2,
           columnspacing=1.2)
savefig(fig, "dose_recovery")

saved dose_recovery (.pdf, .png)


## Fig. targeted harm · group metrics fall, overall AUC does not
The model trained on noisy labels, no repair, scored on the clean test set. Rows = the three main datasets.
* **(a) Detection in the risk group** (recall at the action threshold), all three noise types.
* **(b) Calibration error under targeted noise**: inside the risk group (solid) vs. over all patients (dashed).
  Shown for targeted noise only, because random noise inflates calibration error for *everyone* (it reaches
  0.28-0.31 at the top dose) and would flatten the scale; the point here is that targeted errors leave
  overall calibration untouched and damage only the group's.
* **(c) Overall AUC**, all three noise types: the usual headline metric barely moves under targeted noise.

In [18]:
nr = ver[ver.repair == "none"]
clean = ver[ver.mechanism == "clean"]


def series(d, m, col):
    g = nr[(nr.dataset == d) & (nr.mechanism == m)]
    return g.groupby("dose")[col].agg(["mean", ci]).reset_index()


def band(ax, a, m, **kw):
    ax.plot(a.dose, a["mean"], color=COL[m], marker=MRK[m], ls=LS[m], **kw)
    ax.fill_between(a.dose, a["mean"] - a["ci"], a["mean"] + a["ci"], color=COL[m], alpha=0.15, lw=0)


fig, axes = plt.subplots(len(MAIN), 3, figsize=(PAGE_W, 1.55 * len(MAIN) + 0.1), constrained_layout=True)
for i, d in enumerate(MAIN):
    ax = axes[i, 0]
    for m in MECHS:
        band(ax, series(d, m, "recall_stratum"), m, label=MECH_NAME[m])
    ax.set_ylim(0, None)
    ax.set_ylabel(f"{NAMES[d]}\n\ndetection in risk group")
    ax = axes[i, 1]
    m = "targeted"
    a_g, a_o = series(d, m, "ece_stratum"), series(d, m, "ece")
    ax.plot(a_g.dose, a_g["mean"], color=COL[m], marker=MRK[m], label="risk group")
    ax.fill_between(a_g.dose, a_g["mean"] - a_g["ci"], a_g["mean"] + a_g["ci"], color=COL[m], alpha=0.15, lw=0)
    ax.plot(a_o.dose, a_o["mean"], color=COL[m], ls="--", marker="v", mfc="white", label="all patients")
    ax.fill_between(a_o.dose, a_o["mean"] - a_o["ci"], a_o["mean"] + a_o["ci"], color=COL[m], alpha=0.08, lw=0)
    ax.set_ylim(0, None)
    ax.set_ylabel("calibration error (ECE)")
    r = a_g["mean"].iloc[-1] / a_g["mean"].iloc[0]
    ax.annotate(f"×{r:.1f}", (a_g.dose.iloc[-1], a_g["mean"].iloc[-1]), xytext=(-6, 0),
                textcoords="offset points", ha="right", va="center", color=COL[m], fontsize=7, fontweight="bold")
    ax = axes[i, 2]
    for m in MECHS:
        band(ax, series(d, m, "auc"), m)
    ax.set_ylabel("overall AUC")
for ax in axes.flat:
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4])
for ax in axes[-1]:
    ax.set_xlabel("label-error dose")
for j, t in enumerate(["(a) detection in the risk group", "(b) calibration error (targeted)",
                       "(c) overall AUC"]):
    axes[0, j].set_title(t, loc="left", fontsize=8)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False, handlelength=2.4,
           title="noise type:", title_fontsize=7, alignment="left")
axes[0, 1].legend(frameon=False, loc="upper left", handlelength=2.2)
savefig(fig, "targeted_harm")

saved targeted_harm (.pdf, .png)


## Fig. rate check · aimed at the risk group it detects the damage; run on everyone it does not
Share of the 5 seeds in which the rate check flags the positive rate as too low (p < 0.05), under targeted
noise. Same check, same 400 re-verified records; only the population it is run on differs. Dotted line = the
5% false-alarm rate expected by chance.

In [19]:
ALPHA = 0.05
tg = det[det.mechanism == "targeted"].assign(g=lambda x: x.rate_global_p < ALPHA,
                                             s=lambda x: x.rate_stratum_p < ALPHA)


def rate_panel(ax, d):
    p = tg[tg.dataset == d].groupby("dose")[["g", "s"]].mean()
    ax.axhline(ALPHA, ls=":", lw=0.8, color=GREY)
    ax.plot(p.index, p.s, marker="^", color=COL["targeted"], label="aimed at risk group (Stage 0)")
    ax.plot(p.index, p.g, marker="o", ls="--", mfc="white", color=GREY, label="run on all records")
    ax.set_title(NAMES[d]); ax.set_ylim(-0.04, 1.04); ax.set_xlabel("label-error dose")
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    ax.yaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    ax.set_xticks([0, 0.1, 0.2, 0.3, 0.4])


fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 1.85), sharey=True, constrained_layout=True)
for ax, d in zip(axes, MAIN):
    rate_panel(ax, d)
axes[0].set_ylabel("seeds flagged")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=2, frameon=False, handlelength=2.2)
savefig(fig, "rate_check")
print(tg[tg.dataset.isin(MAIN)].groupby(["dataset", "dose"])[["s", "g"]].mean().unstack(0).round(2))

saved rate_check (.pdf, .png)
                  s                                    g              \
dataset cdcdiabetes diabetes130 taiwancredit cdcdiabetes diabetes130   
dose                                                                   
0.00            0.0         0.0          0.2         0.0         0.2   
0.03            0.0         0.0          0.2         0.0         0.2   
0.05            0.0         0.0          0.2         0.0         0.2   
0.10            0.2         0.0          0.8         0.0         0.2   
0.20            0.8         0.0          1.0         0.0         0.2   
0.30            0.8         0.6          1.0         0.0         0.2   
0.40            1.0         0.8          1.0         0.0         0.2   

                      
dataset taiwancredit  
dose                  
0.00             0.0  
0.03             0.0  
0.05             0.0  
0.10             0.0  
0.20             0.0  
0.30             0.0  
0.40             0.2  


## Fig. repair recovery · five strategies, with their side effects (targeted noise)
Three panels, same five strategies, one bar colour per dataset:
* **(a) Recovery** = share of the *lost* risk-group detection the repair restores (0 = no better than the noisy labels, 1 = back to the clean-label model, below 0 = worse). Open diamonds: the same review with a reviewer right only 80% of the time.
* **(b) Change in overall AUC** vs. no repair. Below 0 = the repair hurt the model for everyone.
* **(c) Change in the risk group's calibration error** vs. no repair. Above 0 = the repair made the group's risk estimates worse.

**Why (b) and (c) are needed.** Recovery alone can mislead. In Taiwan, automatic repair from all-record flags shows high recovery, but confident learning flags about 40% of the *real* defaults as "mislabeled" even when no noise was injected (defaults by clients with good payment histories look surprising). Pruning or relabelling them shifts who gets flagged toward the group and away from everyone else, and lowers AUC and worsens calibration. Panels (b) and (c) make that visible.

Runs: targeted noise, damage ≥ 0.05 (as in notebook 04). Bars = mean ± 95% CI over those runs. Prune and reweight behave like relabel (full table in `04_recovery_recall_stratum.csv`).

In [21]:
r = rec[(rec.mechanism == "targeted")].dropna(subset=["recovery_recall_stratum"]).copy()
base_none = (ver[ver.repair == "none"][["dataset", "seed", "mechanism", "dose", "auc", "ece_stratum"]]
             .rename(columns={"auc": "auc_none", "ece_stratum": "ece_s_none"}))
r = r.merge(base_none, on=["dataset", "seed", "mechanism", "dose"], how="left")
r["d_auc"] = r.auc - r.auc_none                    # < 0: worse for everyone
r["d_ece_s"] = r.ece_stratum - r.ece_s_none        # > 0: group calibration worse

BARS = [  # (label, audit, repair at q=1.0, repair at q=0.8 or None)
    ("Automatic relabel,\nflags from all records", "global", "relabel", None),
    ("Automatic relabel,\nflags from risk group", "stratum", "relabel", None),
    ("Expert review of\nrisk-group flags", "stratum", "review_q1.0", "review_q0.8"),
    ("Expert review, 5% of\ngroup, random order", "budget_random", "review_q1.0", None),
    ("Expert review, 5% of\ngroup, model-ranked", "budget_ranked", "review_q1.0", "review_q0.8"),
]
PANELS = [("recovery_recall_stratum", "(a) Recovery of lost\nrisk-group detection", True),
          ("d_auc", "(b) Change in overall AUC\n(vs. no repair)", False),
          ("d_ece_s", "(c) Change in risk-group\ncalibration error (vs. no repair)", False)]


def stat(d, audit, repair, col):
    s = r[(r.dataset == d) & (r.audit == audit) & (r.repair == repair)][col]
    return s.mean(), ci(s), s.count()


fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 2.75), sharey=True, constrained_layout=True,
                         gridspec_kw={"width_ratios": [1.25, 1, 1]})
y = np.arange(len(BARS))[::-1]
hgt = 0.26
rows = []
for j, (col, title, pct) in enumerate(PANELS):
    ax = axes[j]
    for k, d in enumerate(MAIN):
        off = (1 - k) * hgt
        for yi, (lab, au, rp, rp08) in zip(y, BARS):
            m, e, n = stat(d, au, rp, col)
            ax.barh(yi + off, m, height=hgt, xerr=e, color=DCOL[d], edgecolor="none",
                    error_kw=dict(elinewidth=0.5, capsize=1.2, ecolor="#333333"),
                    label=NAMES[d] if (yi == y[0]) else None)
            if j == 0:
                rows.append((NAMES[d], lab.replace("\n", " "), n))
            if rp08 and j == 0:
                m8, _, _ = stat(d, au, rp08, col)
                ax.plot(m8, yi + off, marker="D", ms=2.8, mfc="white", mec="#333333", mew=0.6, ls="none",
                        label="Reviewer right 80% of the time" if (k == 0 and au == "budget_ranked") else None)
    ax.axvline(0, color="#333333", lw=0.7)
    ax.set_title(title, fontsize=7.5, loc="left")
    ax.grid(axis="y", visible=False)
    if pct:
        ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
axes[0].set_yticks(y); axes[0].set_yticklabels([b[0] for b in BARS], fontsize=6.8)
axes[1].annotate("worse ←", xy=(0.02, -0.16), xycoords="axes fraction", fontsize=6.5, color=GREY)
axes[2].annotate("→ worse", xy=(0.80, -0.16), xycoords="axes fraction", fontsize=6.5, color=GREY)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=4, fontsize=6.8,
           handlelength=1.2, columnspacing=1.0)
savefig(fig, "repair_recovery")

tab = (r.groupby(["dataset", "audit", "repair"])[["recovery_recall_stratum", "d_auc", "d_ece_s"]]
         .mean().round(3).reset_index())
keep = {(au, rp) for _, au, rp, _ in BARS} | {(au, rp8) for _, au, _, rp8 in BARS if rp8}
tab[[(a, b) in keep for a, b in zip(tab.audit, tab.repair)]]

saved repair_recovery (.pdf, .png)


,dataset,audit,repair,recovery_recall_stratum,d_auc,d_ece_s
0,cdcdiabetes,budget_random,review_q1.0,0.071,0.000,-0.002
1,cdcdiabetes,budget_ranked,review_q0.8,0.333,0.001,-0.006
3,cdcdiabetes,budget_ranked,review_q1.0,0.378,0.001,-0.007
5,cdcdiabetes,global,relabel,0.013,-0.001,0.047
8,cdcdiabetes,stratum,relabel,-0.142,-0.005,0.031
9,cdcdiabetes,stratum,review_q0.8,0.096,0.000,-0.001
11,cdcdiabetes,stratum,review_q1.0,0.178,0.001,-0.002
13,diabetes130,budget_random,review_q1.0,0.075,0.001,-0.002
14,diabetes130,budget_ranked,review_q0.8,0.240,0.001,-0.003
16,diabetes130,budget_ranked,review_q1.0,0.287,0.001,-0.004


## Appendix fig. localization · which edge into the label moves
Stage 0 rerun on the noisy labels (seed 0). Each line is one cause of the label: the change in its
standardized coefficient into the label, relative to clean labels. The risk-group variables are coloured (Taiwan: credit limit);
every other cause is grey. Under random noise every edge shrinks together; under targeted noise only the
risk-group edge moves.

In [22]:
FOCUS = {"diabetes130": {"age_years": "#CC3311"},
         "cdcdiabetes": {"NoDocbcCost": "#CC3311", "AnyHealthcare": "#EE7733"},
         "taiwancredit": {"limit_bal": "#CC3311"}}
fig, axes = plt.subplots(len(MAIN), 3, figsize=(PAGE_W, 1.55 * len(MAIN) + 0.1), sharex=True, constrained_layout=True)
for i, d in enumerate(MAIN):
    gd = loc[loc.dataset == d]
    b0 = gd[gd.dose == 0].groupby(["mechanism", "variable"]).coef_into_label.first()
    ever = gd.groupby("variable").coef_into_label.apply(lambda s: s.abs().max())
    vars_ = [v for v in ever.index if ever[v] > 0]
    for j, m in enumerate(MECHS):
        ax = axes[i, j]
        gm = gd[gd.mechanism == m]
        for v in vars_:
            gv = gm[gm.variable == v].sort_values("dose")
            dy = gv.coef_into_label - b0.get((m, v), 0.0)
            if v in FOCUS[d]:
                continue
            ax.plot(gv.dose, dy, color="#BBBBBB", lw=0.8, zorder=1)
        for v, c in FOCUS[d].items():
            gv = gm[gm.variable == v].sort_values("dose")
            ax.plot(gv.dose, gv.coef_into_label - b0.get((m, v), 0.0), color=c, lw=1.8, marker="o", ms=2.5,
                    zorder=3, label=PRETTY.get(v, v))
        ax.axhline(0, color="#333333", lw=0.5)
        ax.set_title(f"{NAMES[d]} · " + {"uniform": "random", "class_conditional": "class-cond.", "targeted": "targeted"}[m], fontsize=7.5, loc="left")
        if j == 0:
            ax.set_ylabel("change in coefficient\ninto the label")
        if i == len(MAIN) - 1:
            ax.set_xlabel("label-error dose")
        ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    h, l = axes[i, 2].get_legend_handles_labels()
    h.append(Line2D([], [], color="#BBBBBB", lw=0.8)); l.append("other causes of the label")
    axes[i, 2].legend(h, l, frameon=False, fontsize=6.5, loc="center left", bbox_to_anchor=(1.0, 0.5),
                      title="risk-group variable" if len(FOCUS[d]) == 1 else "risk-group variables",
                      title_fontsize=6.5, alignment="left")
savefig(fig, "appx_localize")

saved appx_localize (.pdf, .png)


## Appendix fig. South German Credit (retired Oct 10, 2026)
The same three analyses on South German Credit (n = 1,000; risk group = age 25 or under). It was replaced by Taiwan credit because its intervals are too wide and the rate check has too little power; the figure is kept as a record (supplementary material).

In [23]:
for d in [x for x in ("southgerman",) if x in set(det.dataset)]:
    fig, axes = plt.subplots(1, 3, figsize=(PAGE_W, 1.95), constrained_layout=True)
    dose_recovery_panel(axes[0], d)
    for ax in axes:
        ax.set_title("")
    axes[0].set_title("(a) confident-learning estimate", loc="left", fontsize=8)
    axes[0].set_ylabel("estimate above clean floor")
    ax = axes[1]
    for m in MECHS:
        band(ax, series(d, m, "recall_stratum"), m, label=MECH_NAME[m])
    ax.set_ylim(0, None); ax.set_xlabel("label-error dose"); ax.set_ylabel("detection in risk group")
    ax.set_title("(b) detection in the risk group", loc="left", fontsize=8)
    ax.xaxis.set_major_formatter(mpl.ticker.PercentFormatter(1, decimals=0))
    rate_panel(axes[2], d)
    axes[2].set_title("")
    axes[2].set_title("(c) rate check, targeted noise", loc="left", fontsize=8)
    axes[2].set_ylabel("seeds flagged")
    axes[2].legend(frameon=False, fontsize=6.5, loc="upper left")
    h, l = axes[1].get_legend_handles_labels()
    fig.legend(h, l, loc="lower center", bbox_to_anchor=(0.5, 1.0), ncol=3, frameon=False, handlelength=2.4)
    savefig(fig, f"appx_{d}")

saved appx_southgerman (.pdf, .png)


## Numbers for the paper: `li_macros.tex`, and export to the paper folder
**What this does.** Writes every number the paper quotes as a LaTeX macro (`\LI...`), computed here from the CSVs, so the text can never drift from the results. Then copies the figures, these macros, the Stage 0 macros and the ACM class files into the paper folder `TRUST 2027 Papers\label_integrity_paper\`.

Macro names end in the dataset: `Diab` (Diabetes 130), `CDC` (CDC Diabetes), `TWN` (Taiwan credit). Percentages are written without the % sign. **Rerun this section whenever a result changes**, then recompile the paper.

In [ ]:
import shutil, time
sys.path.insert(0, str(ROOT))
import li_lib as li
TAG = {"diabetes130": "Diab", "cdcdiabetes": "CDC", "taiwancredit": "TWN"}
cdir = pd.read_csv(RESULTS / "04b_calibration_direction.csv") if (RESULTS / "04b_calibration_direction.csv").exists() else None
h1fit = pd.read_csv(RESULTS / "03_h1_dose_recovery.csv")
M = {}                                                     # macro name -> text
f2 = lambda x: f"{x:.2f}"; f3 = lambda x: f"{x:.3f}"; pc = lambda x: f"{100 * x:.0f}"
for d, t in TAG.items():
    cfg = li.apply_frozen(li.CONFIGS[d]); tr, te = li.load_split(cfg, "train"), li.load_split(cfg, "test")
    s = li.stratum_mask(tr, cfg)
    M[f"LInTrain{t}"] = f"{len(tr):,}"; M[f"LInTest{t}"] = f"{len(te):,}"
    M[f"LIposRate{t}"] = f2(tr[cfg.label].mean())
    M[f"LIgroupShare{t}"] = pc(s.mean()); M[f"LIgroupPos{t}"] = f"{int(tr.loc[s, cfg.label].sum()):,}"
    M[f"LIgroupPosRate{t}"] = f2(tr.loc[s, cfg.label].mean()); M[f"LIrestPosRate{t}"] = f2(tr.loc[~s, cfg.label].mean())
    # harm (no repair, targeted)
    g = ver[(ver.dataset == d) & (ver.repair == "none") & (ver.mechanism == "targeted")].groupby("dose").mean(numeric_only=True)
    a, b = g.loc[0.0], g.loc[0.4]
    M[f"LIgrpRecallClean{t}"] = f2(a.recall_stratum); M[f"LIgrpRecallTop{t}"] = f2(b.recall_stratum)
    M[f"LIgrpRecallDrop{t}"] = pc(1 - b.recall_stratum / a.recall_stratum)
    M[f"LIrestRecallClean{t}"] = f2(a.recall_rest); M[f"LIrestRecallTop{t}"] = f2(b.recall_rest)
    M[f"LIaucClean{t}"] = f3(a.auc); M[f"LIaucTop{t}"] = f3(b.auc); M[f"LIaucDrop{t}"] = f3(a.auc - b.auc)
    M[f"LIeceGrpClean{t}"] = f3(a.ece_stratum); M[f"LIeceGrpTop{t}"] = f3(b.ece_stratum)
    M[f"LIeceGrpRatio{t}"] = f"{b.ece_stratum / a.ece_stratum:.1f}"
    M[f"LIeceAllClean{t}"] = f3(a.ece); M[f"LIeceAllTop{t}"] = f3(b.ece)
    if cdir is not None:
        c = cdir[(cdir.dataset == d) & (cdir.mechanism == "targeted") & (cdir.dose == 0.4)]
        M[f"LIcalRatioGrp{t}"] = f2(c.group_ratio.mean()); M[f"LIcalRatioRest{t}"] = f2(c.rest_ratio.mean())
    # dose recovery (H1)
    for mech, nm in (("uniform", "Uniform"), ("targeted", "Targeted"), ("class_conditional", "ClassCond")):
        M[f"LIhOne{nm}{t}"] = f2(h1fit[(h1fit.dataset == d) & (h1fit.mechanism == mech)].pearson_r.iloc[0])
    # confident learning at the record level (targeted noise, records inside the group)
    dt = det[(det.dataset == d) & (det.mechanism == "targeted")].groupby("dose").mean(numeric_only=True)
    M[f"LIclRecallTop{t}"] = pc(dt.loc[0.4, "global_in_stratum_recall"])
    M[f"LIclRecallMax{t}"] = pc(dt.loc[dt.index > 0, "global_in_stratum_recall"].max())
    M[f"LIclPrecTop{t}"] = pc(dt.loc[0.4, "global_in_stratum_precision"])
    # confident learning on CLEAN labels: share of real positives it flags as mislabeled
    shares = []
    for seed in li.SEEDS:
        nz = pd.read_csv(RESULTS / "noisy" / f"{d}__targeted__0.00__s{seed}.csv")
        fl = pd.read_csv(RESULTS / "detect" / f"{d}__targeted__0.00__s{seed}.csv").flag_global.astype(bool)
        shares.append(((nz.y_noisy == 1) & fl).sum() / (nz.y_noisy == 1).sum())
    M[f"LIcleanFlagPos{t}"] = pc(np.mean(shares))
    # rate check
    tgd = tg[tg.dataset == d].groupby("dose")[["s", "g"]].mean()
    M[f"LIrateGrpTwenty{t}"] = pc(tgd.loc[0.2, "s"]); M[f"LIrateGrpForty{t}"] = pc(tgd.loc[0.4, "s"])
    first = tgd.index[(tgd.s >= 0.8)]; M[f"LIrateGrpFirstDose{t}"] = pc(first.min()) if len(first) else "--"
    M[f"LIrateAllMax{t}"] = pc(tgd.g.max())
    # repairs (same runs as the repair figure)
    rr = r[r.dataset == d]
    def mean_of(audit, repair, col):
        return rr[(rr.audit == audit) & (rr.repair == repair)][col].mean()
    M[f"LIrecRanked{t}"] = pc(mean_of("budget_ranked", "review_q1.0", "recovery_recall_stratum"))
    M[f"LIrecRankedEighty{t}"] = pc(mean_of("budget_ranked", "review_q0.8", "recovery_recall_stratum"))
    M[f"LIrecRandom{t}"] = pc(mean_of("budget_random", "review_q1.0", "recovery_recall_stratum"))
    M[f"LIrecStratReview{t}"] = pc(mean_of("stratum", "review_q1.0", "recovery_recall_stratum"))
    M[f"LIrecAutoGlobal{t}"] = pc(mean_of("global", "relabel", "recovery_recall_stratum"))
    M[f"LIdAucAutoGlobal{t}"] = f3(mean_of("global", "relabel", "d_auc"))
    M[f"LIdEceAutoGlobal{t}"] = f3(mean_of("global", "relabel", "d_ece_s"))
    M[f"LIrecAutoStratum{t}"] = pc(mean_of("stratum", "relabel", "recovery_recall_stratum"))
    M[f"LIdAucRanked{t}"] = f3(mean_of("budget_ranked", "review_q1.0", "d_auc"))
    M[f"LIdEceRanked{t}"] = f3(mean_of("budget_ranked", "review_q1.0", "d_ece_s"))
M["LIdoses"] = ", ".join(f"{x:g}" for x in li.DOSES); M["LIseeds"] = str(len(li.SEEDS))
M["LIauditSize"] = str(li.AUDIT_SIZE["cdcdiabetes"]); M["LIreviewBudget"] = pc(li.REVIEW_BUDGET)
lines = ["% li_macros.tex -- generated by 05b_paper_figures.ipynb from results/*.csv; do not edit by hand",
         f"% generated {time.strftime('%Y-%m-%d %H:%M')}"]
lines += [f"\\newcommand{{\\{k}}}{{{v}}}" for k, v in M.items()]
(OUT / "li_macros.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")
print(f"wrote {OUT / 'li_macros.tex'} ({len(M)} macros)")

# ---- export to the paper folder -----------------------------------------------------------
PAPER = ROOT.parents[2] / "TRUST 2027 Papers" / "label_integrity_paper"
(PAPER / "figures").mkdir(parents=True, exist_ok=True)
for f in OUT.glob("*.pdf"):
    shutil.copy2(f, PAPER / "figures" / f.name)
shutil.copy2(OUT / "li_macros.tex", PAPER / "li_macros.tex")
for d, t in TAG.items():
    src_ = STAGE0 / d / f"stage0_macros_{t}.tex"
    if src_.exists():
        shutil.copy2(src_, PAPER / src_.name)
ACM = ROOT.parents[2] / "acmart-primary"
for nm in ("acmart.cls", "ACM-Reference-Format.bst"):
    if (ACM / nm).exists() and not (PAPER / nm).exists():
        shutil.copy2(ACM / nm, PAPER / nm)
print("exported figures + macros to", PAPER)

## Check: every figure has a PDF and a PNG

In [24]:
stems = sorted({p.stem for p in OUT.glob("*.pdf")} | {p.stem for p in OUT.glob("*.png")})
check = pd.DataFrame([(s, (OUT / f"{s}.pdf").exists(), (OUT / f"{s}.png").exists()) for s in stems],
                     columns=["figure", "pdf", "png"])
print(check.to_string(index=False))
assert check.pdf.all() and check.png.all(), "a figure is missing a format"

                 figure  pdf  png
          appx_localize True True
       appx_southgerman True True
  calibration_direction True True
          dose_recovery True True
               pipeline True True
             rate_check True True
        repair_recovery True True
 stage0_dag_cdcdiabetes True True
 stage0_dag_diabetes130 True True
stage0_dag_taiwancredit True True
          targeted_harm True True
